# Notebook 01: JEPA Fundamentals — Math & Code From Scratch

**Goal:** Understand the JEPA objective mathematically, then implement a minimal working version.

**Prerequisites:** Basic PyTorch, matrix multiplication, what a neural network is.

---

## 1. What Problem Does JEPA Solve?

Traditional self-supervised learning has two main approaches:

| Approach | What it does | Problem |
|----------|-------------|--------|
| **Generative** (MAE, GPT) | Reconstruct raw pixels/tokens | Wastes capacity on irrelevant details (exact texture, lighting) |
| **Contrastive** (SimCLR, CLIP) | Pull similar views together, push different apart | Needs hand-crafted augmentations, hard negative mining |

**JEPA's insight:** Don't predict pixels. Don't compare views. Instead, **predict abstract representations** of missing content.

```
MAE:   Image → mask patches → predict PIXELS of masked patches
JEPA:  Image → mask patches → predict REPRESENTATIONS of masked patches
```

This lets the model ignore unpredictable details and focus on semantic content.

## 2. The JEPA Objective — Mathematical Formulation

### Core Equation

$$\min_{\theta, \phi, \Delta_y} \| P_\phi(\Delta_y, E_\theta(\mathbf{x})) - \text{sg}(E_{\bar{\theta}}(\mathbf{y})) \|_1$$

Let's break this down piece by piece:

| Symbol | What it is | Role |
|--------|-----------|------|
| $E_\theta$ | **Encoder** (ViT) | Encodes visible (unmasked) patches into representations |
| $E_{\bar{\theta}}$ | **Target encoder** (EMA copy) | Encodes ALL patches to create prediction targets |
| $P_\phi$ | **Predictor** (smaller ViT) | Predicts masked patch representations from visible ones |
| $\Delta_y$ | **Mask tokens** (learnable) | Placeholder tokens at masked positions |
| $\mathbf{x}$ | Visible patches (after masking) | Input to encoder |
| $\mathbf{y}$ | Full image (no masking) | Input to target encoder |
| $\text{sg}(\cdot)$ | **Stop-gradient** | Prevents gradient flow → avoids representation collapse |
| $\|\cdot\|_1$ | **L1 loss** | Mean absolute error in latent space |

### Why stop-gradient + EMA?

Without it, the model could "cheat" by making all representations identical (collapse to zero).
- **Stop-gradient**: Target encoder doesn't receive gradients → can't collapse to match predictions
- **EMA update**: Target encoder slowly tracks the online encoder → stable targets

### EMA Update Rule

$$\bar{\theta} \leftarrow \tau \cdot \bar{\theta} + (1 - \tau) \cdot \theta, \quad \tau = 0.99925$$

This means: each step, the target encoder is 99.925% its old self + 0.075% the current encoder.

## 2.5 The 5W+H of JEPA — Who, What, Where, When, Why, How

### WHO created JEPA?
**Yann LeCun** (Meta/FAIR Chief AI Scientist, Turing Award 2018). Introduced in his 2022 position paper *"A Path Towards Autonomous Machine Intelligence"*. I-JEPA implemented by Assran, Duval, Misra et al. (CVPR 2023). V-JEPA 2 by Bardes, Garrido, Ponce et al. (2025).

### WHAT is JEPA?
A **self-supervised learning architecture** that learns representations by predicting abstract features of masked content from visible content. It consists of:
- An **online encoder** $E_\theta$ (trained via backprop)
- A **target encoder** $E_{\bar\theta}$ (updated via EMA, no gradients)
- A **predictor** $P_\phi$ (trained via backprop)

### WHERE does prediction happen?
**In latent (representation) space**, NOT in pixel/input space. This is the core distinction from MAE/GPT-style models. The loss is computed between D-dimensional vectors, not H×W pixel patches.

### WHEN is each component used?

| Phase | Encoder $E_\theta$ | Target $E_{\bar\theta}$ | Predictor $P_\phi$ |
|-------|-------------------|------------------------|-------------------|
| **Training** | Encodes visible patches | Encodes ALL patches (no grad) | Predicts masked representations |
| **Inference** | Encodes input | Not used | Not used |
| **Robotics (AC)** | Frozen feature extractor | Frozen feature extractor | Action-conditioned world model |

### WHY predict in latent space instead of pixels?

**The information-theoretic argument:**

A 256×256 RGB image has $256 \times 256 \times 3 = 196{,}608$ values.
Most of these encode **textures, lighting, noise** — irrelevant for understanding.

A V-JEPA 2 latent patch is a **1408-dim vector** encoding semantic content.
By predicting here, the model can be **precisely right about semantics** while being **imprecise about irrelevant details**.

Formally, JEPA learns a representation where the **predictable information** (structure, objects, physics) is separated from **unpredictable information** (noise, exact textures).

### HOW does it actually work? (Matrix-level detail)

The complete forward pass as matrix operations:

$$\underbrace{\mathbf{X}}_{\text{image}} \xrightarrow{\text{patchify}} \underbrace{\mathbf{P} \in \mathbb{R}^{N \times d_{\text{patch}}}}_{\text{N patches}} \xrightarrow{\text{mask}} \underbrace{\mathbf{P}_{\text{vis}} \in \mathbb{R}^{M \times d}}_{\text{M visible}}$$

$$\mathbf{P}_{\text{vis}} \xrightarrow{E_\theta} \underbrace{\mathbf{Z}_{\text{vis}} \in \mathbb{R}^{M \times D}}_{\text{encoder output}} \xrightarrow{P_\phi} \underbrace{\hat{\mathbf{H}}_{\text{mask}} \in \mathbb{R}^{(N-M) \times D}}_{\text{predictions for masked}}$$

$$\mathbf{P} \xrightarrow{E_{\bar\theta}} \underbrace{\mathbf{H} \in \mathbb{R}^{N \times D}}_{\text{all targets}} \xrightarrow{\text{gather}} \underbrace{\mathbf{H}_{\text{mask}} \in \mathbb{R}^{(N-M) \times D}}_{\text{targets at masked positions}}$$

$$\mathcal{L} = \frac{1}{(N-M) \cdot D} \sum_{i,j} \left| \hat{\mathbf{H}}_{\text{mask}}[i,j] - \mathbf{H}_{\text{mask}}[i,j] \right|$$

In [ ]:
# MATRIX DIMENSION TRACKING: Step-by-step through a JEPA forward pass
# This shows every tensor shape at every stage

print("=" * 70)
print("JEPA FORWARD PASS — MATRIX DIMENSIONS STEP BY STEP")
print("=" * 70)
print()

# Parameters (V-JEPA 2 real values)
B = 2          # batch size
C = 3          # RGB channels
T = 16         # video frames
H_px = 256     # height in pixels
W_px = 256     # width in pixels
P = 16         # patch size
Tb = 2         # tubelet size (temporal patch)
D = 1408       # ViT-giant embedding dim
D_pred = 384   # predictor embedding dim
N_heads = 22   # attention heads
mask_ratio = 0.90

# Derived
T_tok = T // Tb              # temporal tokens
H_tok = H_px // P            # height tokens
W_tok = W_px // P            # width tokens
N = T_tok * H_tok * W_tok    # total patches
N_vis = int(N * (1 - mask_ratio))
N_mask = N - N_vis
head_dim = D // N_heads

steps = [
    ("INPUT VIDEO",             f"[{B}, {C}, {T}, {H_px}, {W_px}]",
     f"B × C × T × H × W"),
    ("PatchEmbed3D (Conv3d)",   f"[{B}, {D}, {T_tok}, {H_tok}, {W_tok}]",
     f"kernel=({Tb}×{P}×{P}), stride=same"),
    ("flatten + transpose",     f"[{B}, {N}, {D}]",
     f"N = {T_tok}×{H_tok}×{W_tok} = {N} patches"),
    ("MASK (~90%)",             f"vis: [{B}, {N_vis}, {D}]  |  mask: [{B}, {N_mask}, {D}]",
     f"{N_vis} visible ({100*(1-mask_ratio):.0f}%), {N_mask} masked ({100*mask_ratio:.0f}%)"),
    ("Encoder QKV projection",  f"[{B}, {N_vis}, {3*D}] → Q,K,V each [{B}, {N_heads}, {N_vis}, {head_dim}]",
     f"3 × {D} → split into {N_heads} heads of dim {head_dim}"),
    ("3D RoPE rotation",        f"Q,K: [{B}, {N_heads}, {N_vis}, {head_dim}]",
     f"rotate dims [0:{int(2*((head_dim//3)//2))}|{int(2*((head_dim//3)//2))}:{2*int(2*((head_dim//3)//2))}|{2*int(2*((head_dim//3)//2))}:{3*int(2*((head_dim//3)//2))}], unrotated: {head_dim - 3*int(2*((head_dim//3)//2))}"),
    ("Attention: Q·K^T / √d",  f"[{B}, {N_heads}, {N_vis}, {N_vis}]",
     f"softmax over {N_vis} keys per query"),
    ("Attention · V",           f"[{B}, {N_heads}, {N_vis}, {head_dim}]",
     f"weighted sum of values"),
    ("Concat heads + project",  f"[{B}, {N_vis}, {D}]",
     f"concatenate {N_heads} heads → project"),
    ("×40 blocks → Encoder out",f"[{B}, {N_vis}, {D}]",
     f"z_vis: visible patch representations"),
    ("",                        "",
     "────────── PREDICTOR ──────────"),
    ("Project to pred dim",     f"[{B}, {N_vis}, {D_pred}]",
     f"linear {D} → {D_pred}"),
    ("Mask tokens (learnable)",  f"[{B}, {N_mask}, {D_pred}]",
     f"Δ_y: {N_mask} mask tokens, each {D_pred}-dim"),
    ("Concat + sort by position",f"[{B}, {N_vis}+{N_mask}, {D_pred}] = [{B}, {N}, {D_pred}]",
     f"all {N} tokens in position order"),
    ("×12 blocks → Predictor",  f"[{B}, {N}, {D_pred}]",
     f"all tokens processed jointly"),
    ("Extract mask tokens",     f"[{B}, {N_mask}, {D_pred}]",
     f"reverse sort, take mask positions only"),
    ("Project back",            f"[{B}, {N_mask}, {D}]",
     f"linear {D_pred} → {D}: predictions ẑ_mask"),
    ("",                        "",
     "────────── TARGET ──────────"),
    ("Target enc (ALL patches)",f"[{B}, {N}, {D}]",
     f"EMA encoder processes all {N} patches (no mask)"),
    ("LayerNorm targets",       f"[{B}, {N}, {D}]",
     f"normalize target representations"),
    ("Gather at mask positions",f"[{B}, {N_mask}, {D}]",
     f"h_mask: ground-truth targets"),
    ("",                        "",
     "────────── LOSS ──────────"),
    ("L1 loss: |ẑ_mask - h_mask|",f"scalar",
     f"mean over [{B}, {N_mask}, {D}] = {B*N_mask*D:,} values"),
]

print(f"{'Step':<30} {'Shape':<55} {'Notes'}")
print("-" * 140)
for step, shape, notes in steps:
    if step == "":
        print(f"\n{notes}")
    else:
        print(f"{step:<30} {shape:<55} {notes}")

## 3. Let's Implement It — Minimal JEPA on Toy Data

We'll build a tiny JEPA from scratch using only basic PyTorch. No ViT needed yet — just MLPs to understand the flow.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import copy

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: CPU (AMD Radeon — we'll use CPU for study)")

### 3.1 Create Toy "Image" Data

We'll represent an "image" as a sequence of 16 patches, each a 32-dim vector.
Think of this as a 4×4 grid of patches from a 64×64 image with patch_size=16.

In [ ]:
# Toy dataset: 100 "images", each with 16 patches of dimension 32
N_images = 200
N_patches = 16      # 4x4 grid
patch_dim = 32      # each patch is a 32-dim vector

# Create structured data (not pure noise — patches near each other are correlated)
data = []
for i in range(N_images):
    # Base pattern + local variation (simulates spatial correlation in real images)
    base = torch.randn(1, patch_dim) * 2  # global image-level pattern
    patches = base.repeat(N_patches, 1) + torch.randn(N_patches, patch_dim) * 0.5
    data.append(patches)
data = torch.stack(data)  # [200, 16, 32]

print(f"Dataset shape: {data.shape}  (N_images, N_patches, patch_dim)")
print(f"This represents {N_images} images, each split into {N_patches} patches of dim {patch_dim}")

### 3.2 Masking Strategy

JEPA masks ~75-90% of patches. The encoder only sees the visible patches.
The predictor must predict representations of the masked patches.

V-JEPA 2 uses **~90% masking** with spatiotemporal blocks. For simplicity, we'll do random masking.

In [ ]:
def create_masks(batch_size, n_patches, mask_ratio=0.75):
    """
    Create random masks for JEPA.
    Returns:
        visible_indices: indices of patches the encoder sees
        masked_indices:  indices of patches the predictor must predict
    """
    n_masked = int(n_patches * mask_ratio)
    n_visible = n_patches - n_masked
    
    visible_list = []
    masked_list = []
    for _ in range(batch_size):
        perm = torch.randperm(n_patches)
        visible_list.append(perm[:n_visible].sort()[0])
        masked_list.append(perm[n_visible:].sort()[0])
    
    return torch.stack(visible_list), torch.stack(masked_list)

# Demo
vis_idx, mask_idx = create_masks(batch_size=2, n_patches=16, mask_ratio=0.75)
print(f"Visible patches (encoder sees):  {vis_idx[0].tolist()}")
print(f"Masked patches (predictor predicts): {mask_idx[0].tolist()}")
print(f"\nEncoder sees {vis_idx.shape[1]}/{N_patches} = {vis_idx.shape[1]/N_patches*100:.0f}% of patches")
print(f"Predictor must predict {mask_idx.shape[1]}/{N_patches} = {mask_idx.shape[1]/N_patches*100:.0f}% of patches")

### 3.3 Build the Three Components

In real V-JEPA 2, these are Vision Transformers. For understanding, we use simple MLPs.

In [ ]:
class ToyEncoder(nn.Module):
    """Maps patch tokens → latent representations.
    
    In V-JEPA 2, this is a ViT (e.g., ViT-giant: 1408-dim, 40 layers, 22 heads).
    Here we use a simple 2-layer MLP to understand the concept.
    """
    def __init__(self, input_dim=32, hidden_dim=64, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
            nn.LayerNorm(output_dim),  # V-JEPA 2 applies LayerNorm to target encoder outputs
        )
    
    def forward(self, x):
        # x: [B, N_visible, input_dim]
        return self.net(x)  # [B, N_visible, output_dim]


class ToyPredictor(nn.Module):
    """Predicts representations of masked patches from visible patch representations.
    
    In V-JEPA 2, this is a smaller ViT (12 layers, 384-dim, 12 heads).
    Key components:
    - Projects encoder output to predictor dimension
    - Learnable MASK TOKENS placed at target positions
    - Projects back to encoder dimension
    """
    def __init__(self, encoder_dim=64, predictor_dim=32, n_patches=16):
        super().__init__()
        # Project encoder dim → predictor dim (smaller)
        self.embed = nn.Linear(encoder_dim, predictor_dim)
        
        # Learnable mask token — placed at masked positions
        # This is Δ_y in the paper
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        
        # Positional embeddings (so the predictor knows WHERE each patch is)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, predictor_dim) * 0.02)
        
        # Processing layers
        self.net = nn.Sequential(
            nn.Linear(predictor_dim, predictor_dim * 2),
            nn.GELU(),
            nn.Linear(predictor_dim * 2, predictor_dim),
        )
        
        # Project back to encoder dimension
        self.proj = nn.Linear(predictor_dim, encoder_dim)
    
    def forward(self, z_visible, visible_idx, masked_idx):
        """
        z_visible: [B, N_visible, encoder_dim] — encoder output for visible patches
        visible_idx: [B, N_visible] — which patch positions are visible
        masked_idx: [B, N_masked] — which patch positions are masked
        Returns: [B, N_masked, encoder_dim] — predicted representations for masked patches
        """
        B = z_visible.shape[0]
        N_masked = masked_idx.shape[1]
        
        # Step 1: Project visible tokens to predictor dimension
        z = self.embed(z_visible)  # [B, N_visible, predictor_dim]
        
        # Step 2: Add positional embeddings to visible tokens
        vis_pos = self.pos_embed[:, :, :].expand(B, -1, -1)  # [B, N_patches, D]
        vis_pos_gathered = torch.stack([
            vis_pos[b, visible_idx[b]] for b in range(B)
        ])  # [B, N_visible, D]
        z = z + vis_pos_gathered
        
        # Step 3: Create mask tokens with positional embeddings at masked positions
        mask_tokens = self.mask_token.expand(B, N_masked, -1)  # [B, N_masked, D]
        mask_pos = torch.stack([
            vis_pos[b, masked_idx[b]] for b in range(B)
        ])  # [B, N_masked, D]
        mask_tokens = mask_tokens + mask_pos
        
        # Step 4: Concatenate [visible + mask_tokens], process, extract masked
        x = torch.cat([z, mask_tokens], dim=1)  # [B, N_visible + N_masked, D]
        x = self.net(x)
        
        # Step 5: Extract only the mask token outputs and project back
        x_masked = x[:, z.shape[1]:]  # [B, N_masked, D]
        x_masked = self.proj(x_masked)  # [B, N_masked, encoder_dim]
        
        return x_masked


# Instantiate
encoder = ToyEncoder(input_dim=patch_dim, hidden_dim=64, output_dim=64)
predictor = ToyPredictor(encoder_dim=64, predictor_dim=32, n_patches=N_patches)
target_encoder = copy.deepcopy(encoder)  # EMA copy

# Freeze target encoder (no gradients)
for p in target_encoder.parameters():
    p.requires_grad = False

print(f"Encoder params:        {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Predictor params:      {sum(p.numel() for p in predictor.parameters()):,}")
print(f"Target encoder params: {sum(p.numel() for p in target_encoder.parameters()):,} (frozen)")

### 3.4 The Training Loop — JEPA in Action

Here's the complete JEPA training flow:

```
1. Mask patches randomly
2. Target encoder: encode ALL patches → target representations (no gradient)
3. Encoder: encode VISIBLE patches only → context representations
4. Predictor: predict MASKED patch representations from context
5. L1 Loss between predictions and targets at masked positions
6. Backprop through encoder + predictor (NOT target encoder)
7. EMA update target encoder
```

In [ ]:
# Training hyperparameters (matching V-JEPA 2)
EMA_MOMENTUM = 0.99925       # tau — how much target encoder tracks online encoder
MASK_RATIO = 0.75            # V-JEPA 2 uses ~0.90, we use 0.75 for faster convergence on toy data
LOSS_EXP = 1.0               # L1 loss (exponent=1)
LR = 1e-3
EPOCHS = 50
BATCH_SIZE = 32

optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(predictor.parameters()),
    lr=LR, weight_decay=0.04  # V-JEPA 2 uses wd=0.04
)

losses = []

for epoch in range(EPOCHS):
    epoch_loss = 0
    n_batches = 0
    
    # Shuffle data
    perm = torch.randperm(len(data))
    
    for i in range(0, len(data) - BATCH_SIZE, BATCH_SIZE):
        batch = data[perm[i:i+BATCH_SIZE]]  # [B, 16, 32]
        B = batch.shape[0]
        
        # Step 1: Create masks
        visible_idx, masked_idx = create_masks(B, N_patches, MASK_RATIO)
        
        # Step 2: Target encoder — encode ALL patches (no gradient!)
        with torch.no_grad():
            h_full = target_encoder(batch)  # [B, 16, 64]
            # Apply layer_norm (as V-JEPA 2 does)
            h_full = F.layer_norm(h_full, (h_full.size(-1),))
            # Extract target representations at masked positions
            h_target = torch.stack([h_full[b, masked_idx[b]] for b in range(B)])  # [B, N_masked, 64]
        
        # Step 3: Encoder — encode VISIBLE patches only
        x_visible = torch.stack([batch[b, visible_idx[b]] for b in range(B)])  # [B, N_visible, 32]
        z_visible = encoder(x_visible)  # [B, N_visible, 64]
        
        # Step 4: Predictor — predict masked representations
        z_predicted = predictor(z_visible, visible_idx, masked_idx)  # [B, N_masked, 64]
        
        # Step 5: L1 Loss
        loss = torch.mean(torch.abs(z_predicted - h_target) ** LOSS_EXP) / LOSS_EXP
        
        # Step 6: Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Step 7: EMA update of target encoder
        with torch.no_grad():
            for param_q, param_k in zip(encoder.parameters(), target_encoder.parameters()):
                param_k.data.mul_(EMA_MOMENTUM).add_(param_q.data, alpha=1.0 - EMA_MOMENTUM)
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / max(n_batches, 1)
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")

In [ ]:
# EMA CONVERGENCE VISUALIZATION
# Shows how the target encoder slowly tracks the online encoder over training

import numpy as np
import copy

def visualize_ema_convergence():
    """Demonstrate EMA convergence: target encoder tracks online encoder."""
    
    # Simulate 1000 training steps
    n_steps = 1000
    tau = 0.99925  # V-JEPA 2's momentum
    
    # Track a single parameter value
    online_param = 0.0
    target_param = 0.0
    
    online_history = []
    target_history = []
    
    for step in range(n_steps):
        # Simulate online encoder parameter changing (random walk + drift)
        online_param += np.random.randn() * 0.1 + 0.01  # upward drift
        
        # EMA update
        target_param = tau * target_param + (1 - tau) * online_param
        
        online_history.append(online_param)
        target_history.append(target_param)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Plot 1: Full trajectory
    axes[0].plot(online_history, 'b-', alpha=0.5, linewidth=1, label='Online encoder θ')
    axes[0].plot(target_history, 'r-', linewidth=2, label=f'Target encoder θ̄ (τ={tau})')
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('Parameter Value')
    axes[0].set_title('EMA: Target Slowly Tracks Online')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Different tau values
    for tau_val, color, style in [(0.99, '#e74c3c', '-'), (0.999, '#3498db', '-'), 
                                   (0.99925, '#2ecc71', '-'), (0.9999, '#9b59b6', '-')]:
        target = 0.0
        targets = []
        for step in range(n_steps):
            target = tau_val * target + (1 - tau_val) * online_history[step]
            targets.append(target)
        axes[1].plot(targets, color=color, linewidth=1.5, label=f'τ={tau_val}')
    
    axes[1].plot(online_history, 'k-', alpha=0.2, linewidth=1, label='Online (reference)')
    axes[1].set_xlabel('Training Step')
    axes[1].set_title('Effect of EMA Momentum τ')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Effective averaging window
    taus = np.linspace(0.9, 0.9999, 100)
    windows = 1 / (1 - taus)
    axes[2].semilogy(taus, windows, 'b-', linewidth=2)
    axes[2].axvline(x=0.99925, color='r', linestyle='--', label=f'V-JEPA 2: τ=0.99925\nwindow≈{1/(1-0.99925):.0f} steps')
    axes[2].set_xlabel('Momentum τ')
    axes[2].set_ylabel('Effective Averaging Window (steps)')
    axes[2].set_title('EMA = Exponential Moving Average')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"V-JEPA 2 uses τ = 0.99925")
    print(f"Effective averaging window: 1/(1-τ) = {1/(1-0.99925):.0f} steps")
    print(f"This means the target encoder 'remembers' roughly the last {1/(1-0.99925):.0f} updates")
    print(f"At batch_size=3072 and ~400K steps, this covers ~1333/400000 ≈ 0.33% of training")

visualize_ema_convergence()

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('L1 Loss (in latent space)')
plt.title('JEPA Training Loss — Predicting Representations, NOT Pixels')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.6 What Happens WITHOUT Stop-Gradient? (Representation Collapse)

The most important design choice in JEPA is the **stop-gradient** on the target encoder. Without it, the model finds a trivial solution: **make all representations identical (constant zero)**.

**Why collapse happens (mathematical argument):**

The loss is $\mathcal{L} = \| \hat{z} - z_{\text{target}} \|_1$. If both encoder and target encoder get gradients:
- The easiest way to minimize the loss is for **both** to output the same constant
- $E_\theta(x) = \mathbf{0}$ for all $x$, and $E_{\bar\theta}(y) = \mathbf{0}$ for all $y$
- Then $\mathcal{L} = \|\mathbf{0} - \mathbf{0}\| = 0$ — perfect loss, useless representations!

**Stop-gradient breaks this:**
- Target encoder doesn't receive gradients → it can't move toward zero
- Online encoder must learn to actually predict target encoder's output
- EMA gives the target encoder slow, stable movement → meaningful learning signal

### 3.5 Verify: The Encoder Learned Meaningful Representations

If JEPA worked, similar images should have similar representations.

In [ ]:
# REPRESENTATION COLLAPSE EXPERIMENT
# Train the SAME toy JEPA but WITHOUT stop-gradient — watch it collapse!

encoder_collapse = ToyEncoder(input_dim=patch_dim, hidden_dim=64, output_dim=64)
predictor_collapse = ToyPredictor(encoder_dim=64, predictor_dim=32, n_patches=N_patches)
target_collapse = copy.deepcopy(encoder_collapse)
# NOTE: target parameters ARE trainable (no stop-gradient!)

optimizer_collapse = torch.optim.AdamW(
    list(encoder_collapse.parameters()) + list(predictor_collapse.parameters()) + list(target_collapse.parameters()),
    lr=1e-3, weight_decay=0.04
)

losses_collapse = []
rep_norms_collapse = []  # track representation magnitudes

for epoch in range(50):
    epoch_loss = 0
    n_batches = 0
    perm = torch.randperm(len(data))
    
    for i in range(0, len(data) - BATCH_SIZE, BATCH_SIZE):
        batch = data[perm[i:i+BATCH_SIZE]]
        B_local = batch.shape[0]
        visible_idx, masked_idx = create_masks(B_local, N_patches, MASK_RATIO)
        
        # Target encoder — NO stop-gradient! Gradients flow through!
        h_full = target_collapse(batch)
        h_full_normed = F.layer_norm(h_full, (h_full.size(-1),))
        h_target = torch.stack([h_full_normed[b, masked_idx[b]] for b in range(B_local)])
        
        # Encoder
        x_visible = torch.stack([batch[b, visible_idx[b]] for b in range(B_local)])
        z_visible = encoder_collapse(x_visible)
        
        # Predictor
        z_predicted = predictor_collapse(z_visible, visible_idx, masked_idx)
        
        # Loss — gradient flows to BOTH encoder AND target encoder
        loss = torch.mean(torch.abs(z_predicted - h_target))
        
        optimizer_collapse.zero_grad()
        loss.backward()
        optimizer_collapse.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / max(n_batches, 1)
    losses_collapse.append(avg_loss)
    
    # Track representation norms
    with torch.no_grad():
        rep = target_collapse(data[:10])
        rep_norms_collapse.append(rep.norm(dim=-1).mean().item())

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss comparison
axes[0].plot(losses, 'b-', linewidth=2, label='With stop-gradient (correct)')
axes[0].plot(losses_collapse, 'r-', linewidth=2, label='WITHOUT stop-gradient (collapse)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('L1 Loss')
axes[0].set_title('Loss: Collapse = Deceptively Low Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Representation norms
axes[1].plot(rep_norms_collapse, 'r-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Representation L2 Norm')
axes[1].set_title('WITHOUT Stop-Gradient:\nRepresentations Shrink Toward Zero')
axes[1].grid(True, alpha=0.3)

# Similarity matrix for collapsed model
with torch.no_grad():
    reps_col = encoder_collapse(data)
    global_reps_col = reps_col.mean(dim=1)
    norms = global_reps_col.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    global_reps_col_norm = global_reps_col / norms
    sim_col = global_reps_col_norm @ global_reps_col_norm.T

axes[2].imshow(sim_col[:50, :50].numpy(), cmap='viridis', vmin=-1, vmax=1)
axes[2].set_title('Collapsed Representations:\nAll images look "similar" (useless)')
axes[2].set_xlabel('Image index')
axes[2].set_ylabel('Image index')

plt.tight_layout()
plt.show()

print("KEY TAKEAWAY:")
print("Without stop-gradient, the loss goes to ~0 but representations COLLAPSE.")
print("All images get mapped to nearly the same point → useless features.")
print("Stop-gradient + EMA prevents this by keeping the target encoder stable.")

In [ ]:
# Encode all images using the trained encoder
with torch.no_grad():
    all_reps = encoder(data)  # [200, 16, 64]
    # Global representation: mean pool over patches
    global_reps = all_reps.mean(dim=1)  # [200, 64]

# Compute pairwise cosine similarity
global_reps_norm = F.normalize(global_reps, dim=-1)
sim_matrix = global_reps_norm @ global_reps_norm.T

plt.figure(figsize=(6, 5))
plt.imshow(sim_matrix[:50, :50].numpy(), cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.title('Learned Representation Similarity (first 50 images)')
plt.xlabel('Image index')
plt.ylabel('Image index')
plt.tight_layout()
plt.show()

print(f"Mean self-similarity: {sim_matrix.diag().mean():.3f} (should be 1.0)")
print(f"Mean cross-similarity: {(sim_matrix.sum() - sim_matrix.diag().sum()) / (200*199):.3f}")

## 4. Key Differences: V-JEPA 2 vs Our Toy Model

| Component | Our Toy Model | V-JEPA 2 (Real) |
|-----------|--------------|------------------|
| **Encoder** | 2-layer MLP | ViT-giant (1408-dim, 40 layers, 22 heads, ~1B params) |
| **Predictor** | 2-layer MLP | ViT-small-like (384-dim, 12 layers, 12 heads) |
| **Input** | Random 32-dim vectors | Video frames → 3D tubelets (2×16×16) |
| **Masking** | Random 75% | Spatiotemporal blocks, ~90% masked |
| **Position encoding** | Learnable 1D | 3D Rotary Position Embeddings (depth/height/width) |
| **EMA momentum** | 0.99925 (fixed) | 0.99925 (fixed) — same! |
| **Loss** | L1 | L1 — same! |
| **Training data** | 200 toy vectors | 1M+ hours of internet video |
| **Training compute** | Seconds on CPU | 128 GPUs for 800 epochs |

The **mathematical framework is identical**. The difference is scale and the use of Vision Transformers.

## 5. Deep Dive: Why L1 Loss and Not L2?

V-JEPA 2 uses `loss_exp = 1.0` → pure L1 (Mean Absolute Error).

From the V-JEPA 2 training code (`app/vjepa/train.py`, line 447):
```python
loss += torch.mean(torch.abs(zij - hij) ** loss_exp) / loss_exp
```

**Why L1 over L2?**
- L2 loss squares errors → large errors dominate → model focuses on outliers
- L1 loss is linear → more robust to unpredictable patches
- JEPA wants to be imprecise about unpredictable content → L1 doesn't over-penalize that

Let's visualize the difference:

In [ ]:
# L1 vs L2 loss comparison
x = torch.linspace(-3, 3, 100)
l1 = torch.abs(x)
l2 = x ** 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x.numpy(), l1.numpy(), 'b-', linewidth=2, label='L1: |x|')
ax1.plot(x.numpy(), l2.numpy(), 'r-', linewidth=2, label='L2: x²')
ax1.set_xlabel('Prediction Error')
ax1.set_ylabel('Loss')
ax1.set_title('L1 vs L2 Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 5)

# Gradient comparison
l1_grad = torch.sign(x)  # gradient of L1 is ±1 everywhere
l2_grad = 2 * x  # gradient of L2 is 2x → large for large errors

ax2.plot(x.numpy(), l1_grad.numpy(), 'b-', linewidth=2, label='L1 gradient: sign(x)')
ax2.plot(x.numpy(), l2_grad.numpy(), 'r-', linewidth=2, label='L2 gradient: 2x')
ax2.set_xlabel('Prediction Error')
ax2.set_ylabel('Gradient Magnitude')
ax2.set_title('Gradient: L1 is constant, L2 explodes for large errors')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: L1 treats all prediction errors equally.")
print("L2 would make the model obsess over hard-to-predict patches (noise, texture).")
print("JEPA WANTS to ignore unpredictable details → L1 is the right choice.")

## 6.5 Deep Dive: The `apply_masks` Gather Operation

In the real V-JEPA 2 code, masking is done via `torch.gather` — an efficient indexing operation:

```python
# From refs/vjepa2/src/models/vision_transformer.py, line 198-201
def apply_masks(x, masks):
    all_x = []
    for m in masks:
        mask_keep = m.unsqueeze(-1).repeat(1, 1, x.size(-1))
        all_x.append(torch.gather(x, dim=1, index=mask_keep))
    return torch.cat(all_x, dim=0)
```

**What `torch.gather` does (matrix indexing):**

Given `x` of shape `[B, N, D]` and `mask` of shape `[B, M]`:

$$\text{gather}(\mathbf{X}, \mathbf{m})_{b,i,d} = \mathbf{X}_{b, \mathbf{m}_{b,i}, d}$$

This selects `M` rows from the `N` patches for each batch element. It's equivalent to fancy indexing but GPU-optimized.

**Example with N=8 patches, keeping M=2:**
```
x = [[x₀, x₁, x₂, x₃, x₄, x₅, x₆, x₇]]     # [1, 8, D]
mask = [[2, 5]]                                    # keep patches 2 and 5
gather(x, mask) = [[x₂, x₅]]                      # [1, 2, D]
```

This is much faster than Python loops because it runs entirely on GPU in a single kernel.

## 6. Deep Dive: The Masking Strategy

V-JEPA 2 uses **multiblock spatiotemporal masking** — not random masking!

From `src/masks/multiseq_multiblock3d.py`:

```
Config 1: 8 small blocks (15% spatial area each, full temporal span)
Config 2: 2 large blocks (70% spatial area each, full temporal span)
→ Union of all blocks → ~90% of patches masked
```

**Critical design:** All blocks span the **full temporal dimension**. This prevents the model from simply copying adjacent frames and forces it to learn spatial prediction.

In [ ]:
# GATHER OPERATION VISUALIZATION + Information Theory of Masking
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LEFT: Visualize gather operation
ax = axes[0]
N_demo, D_demo = 8, 4
x_demo = torch.arange(N_demo * D_demo).float().reshape(1, N_demo, D_demo)
mask_demo = torch.tensor([[1, 3, 6]])  # keep patches 1, 3, 6

mask_expanded = mask_demo.unsqueeze(-1).expand(-1, -1, D_demo)
gathered = torch.gather(x_demo, dim=1, index=mask_expanded)

# Show as heatmap
full_data = x_demo[0].numpy()
gathered_data = gathered[0].numpy()

# Full matrix with highlights
ax.imshow(full_data, cmap='Blues', aspect='auto')
for i in range(N_demo):
    for j in range(D_demo):
        color = 'white' if i in [1, 3, 6] else 'black'
        weight = 'bold' if i in [1, 3, 6] else 'normal'
        ax.text(j, i, f'{full_data[i, j]:.0f}', ha='center', va='center', 
                fontsize=8, color=color, fontweight=weight)

for idx in [1, 3, 6]:
    ax.add_patch(plt.Rectangle((-0.5, idx-0.5), D_demo, 1, fill=False, 
                                edgecolor='red', linewidth=2))

ax.set_xlabel('Embedding dimension d')
ax.set_ylabel('Patch index n')
ax.set_title('torch.gather: Select rows by index\n(Red boxes = selected patches 1, 3, 6)')
ax.set_yticks(range(N_demo))
ax.set_yticklabels([f'patch {i}' for i in range(N_demo)])

# RIGHT: Information theory — why ~90% masking?
ax = axes[1]

mask_ratios = np.linspace(0.1, 0.99, 50)
# Difficulty: higher mask ratio = harder prediction task = richer learning signal
difficulty = mask_ratios  # linear with mask ratio
# Visible context: lower mask ratio = more context = easier to make correct predictions
context = 1 - mask_ratios
# Sweet spot: product of difficulty × context quality
# (need enough context to learn, but enough masking to make it challenging)
quality = difficulty * context * 4  # normalize to peak at ~1.0

ax.plot(mask_ratios * 100, difficulty, 'r-', linewidth=2, label='Task difficulty (higher = harder)')
ax.plot(mask_ratios * 100, context, 'b-', linewidth=2, label='Context available (higher = more info)')
ax.plot(mask_ratios * 100, quality, 'g-', linewidth=3, label='Learning signal quality')
ax.axvline(x=90, color='purple', linestyle='--', linewidth=2, label='V-JEPA 2: 90% mask')
ax.axvline(x=75, color='orange', linestyle='--', linewidth=2, label='MAE/I-JEPA: 75% mask')

ax.set_xlabel('Mask Ratio (%)')
ax.set_ylabel('Relative Magnitude')
ax.set_title('Why ~90% Masking? Information-Theoretic View')
ax.legend(fontsize=8, loc='center right')
ax.grid(True, alpha=0.3)
ax.set_xlim(10, 99)

plt.tight_layout()
plt.show()

print("Why V-JEPA 2 uses ~90% masking (higher than MAE's 75%):")
print("• Video has MORE temporal redundancy than images → need higher masking")
print("• Full-temporal-span blocks prevent temporal copying → forces spatial reasoning")
print("• ~90% is the sweet spot: hard enough to learn, but enough context to succeed")
print("• Also: only processing 10% of tokens makes the encoder 10x more efficient!")

In [ ]:
def visualize_jepa_masking(H=16, W=16, T=8):
    """
    Simulate V-JEPA 2's multiblock masking strategy.
    H, W: spatial grid (16x16 for 256px with patch_size=16)
    T: temporal frames (tubelets)
    """
    mask = torch.ones(T, H, W)
    
    # Config 1: 8 small blocks (15% spatial scale)
    for _ in range(8):
        # 15% of H*W = ~0.15 * 256 = 38 patches
        n_keep = int(H * W * 0.15)
        aspect = 0.75 + torch.rand(1).item() * 0.75  # [0.75, 1.5]
        bh = min(int(round((n_keep * aspect) ** 0.5)), H)
        bw = min(int(round((n_keep / aspect) ** 0.5)), W)
        top = torch.randint(0, max(1, H - bh + 1), (1,)).item()
        left = torch.randint(0, max(1, W - bw + 1), (1,)).item()
        mask[:, top:top+bh, left:left+bw] = 0  # Full temporal span!
    
    # Config 2: 2 large blocks (70% spatial scale)
    for _ in range(2):
        n_keep = int(H * W * 0.70)
        aspect = 0.75 + torch.rand(1).item() * 0.75
        bh = min(int(round((n_keep * aspect) ** 0.5)), H)
        bw = min(int(round((n_keep / aspect) ** 0.5)), W)
        top = torch.randint(0, max(1, H - bh + 1), (1,)).item()
        left = torch.randint(0, max(1, W - bw + 1), (1,)).item()
        mask[:, top:top+bh, left:left+bw] = 0
    
    return mask

# Visualize
mask = visualize_jepa_masking()
total_patches = mask.numel()
visible_patches = mask.sum().item()
masked_patches = total_patches - visible_patches

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for t in range(8):
    ax = axes[t // 4][t % 4]
    ax.imshow(mask[t].numpy(), cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_title(f'Frame {t}')
    ax.axis('off')

plt.suptitle(
    f'V-JEPA 2 Masking: {visible_patches}/{total_patches} visible '
    f'({visible_patches/total_patches*100:.0f}%), '
    f'{masked_patches}/{total_patches} masked '
    f'({masked_patches/total_patches*100:.0f}%)\n'
    f'Green = visible (encoder sees), Red = masked (predictor predicts)',
    fontsize=12
)
plt.tight_layout()
plt.show()

print(f"Notice: The mask is THE SAME across all frames (full temporal span).")
print(f"This forces spatial prediction, not temporal copying.")

## 7. Summary: JEPA Core Equations

### The Complete JEPA System

**Pretraining Objective:**
$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \left| P_\phi(\Delta_y, E_\theta(\mathbf{x}_i)) - \text{sg}(E_{\bar{\theta}}(\mathbf{y}_i)) \right|$$

**EMA Update:**
$$\bar{\theta} \leftarrow 0.99925 \cdot \bar{\theta} + 0.00075 \cdot \theta$$

**Masking (V-JEPA 2):**
- 8 small blocks (15% spatial, full temporal) + 2 large blocks (70% spatial, full temporal)
- Result: ~90% patches masked, ~10% visible

**Key Design Choices:**
1. **Predict in latent space** (not pixel space) → ignores irrelevant details
2. **L1 loss** → robust to unpredictable content
3. **EMA target encoder** → stable targets, prevents collapse
4. **Stop-gradient on targets** → prevents trivial solution
5. **Full temporal span masking** → forces spatial understanding

---

**Next notebook:** We'll look at the actual V-JEPA 2 ViT architecture, 3D RoPE, and how the real encoder/predictor work.